In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.74MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.05MB/s]


In [4]:
train_dataloader = DataLoader(training_data, batch_size = 64)
test_dataloader = DataLoader(test_data, batch_size= 64)

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [7]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [8]:
loss_fn = nn.CrossEntropyLoss()

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred,y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [15]:
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches 
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")     

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

epochs = 10

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.904205  [   64/60000]
loss: 1.874529  [ 6464/60000]
loss: 1.745435  [12864/60000]
loss: 1.800479  [19264/60000]
loss: 1.673090  [25664/60000]
loss: 1.632304  [32064/60000]
loss: 1.659193  [38464/60000]
loss: 1.560247  [44864/60000]
loss: 1.587345  [51264/60000]
loss: 1.495698  [57664/60000]
Test Error: 
 Accuracy: 61.0%, Avg loss: 1.498688 

Epoch 2
-------------------------------
loss: 1.564684  [   64/60000]
loss: 1.533533  [ 6464/60000]
loss: 1.375126  [12864/60000]
loss: 1.461373  [19264/60000]
loss: 1.328492  [25664/60000]
loss: 1.334026  [32064/60000]
loss: 1.347320  [38464/60000]
loss: 1.274546  [44864/60000]
loss: 1.310083  [51264/60000]
loss: 1.223087  [57664/60000]
Test Error: 
 Accuracy: 63.3%, Avg loss: 1.238196 

Epoch 3
-------------------------------
loss: 1.314429  [   64/60000]
loss: 1.300348  [ 6464/60000]
loss: 1.127767  [12864/60000]
loss: 1.244967  [19264/60000]
loss: 1.109581  [25664/60000]
loss: 1.141984  [32064/600